In [2]:
# %% [markdown]
# # NB_03 — Ingestion des arrêts de transport en France (Microsoft Fabric)
# 
# **Workspace :** `Mobility_Intelligence_AI`  
# **Lakehouse par défaut :** `lh_mobility_citiz`
# 
# Télécharge le CSV national officiel dans `Files/bronze/gtfs_national`, contrôle son intégrité
# et sa structure, documente la provenance, puis prépare une table Delta Silver.
# 
# **Important :** l'agrégation nationale est expérimentale : arrêts parfois en doublon,
# `stop_id` non unique, date de publication potentiellement ancienne. Les DROM peuvent être
# présents, mais leur couverture exhaustive n'est pas garantie. Ce fichier ne contient
# **pas** les réseaux piétons ni les fréquences de desserte.
# 
# Source : https://www.data.gouv.fr/datasets/arrets-de-transport-en-france

# %%
# 1 — Configuration et vérification de Fabric
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import requests

LAKEHOUSE_FILES = Path('/lakehouse/default/Files')
if not LAKEHOUSE_FILES.is_dir():
    raise RuntimeError(
        'Lakehouse absent : ajouter lh_mobility_citiz au Notebook et le définir par défaut, '
        'puis redémarrer la session Spark.'
    )

DATASET_API = 'https://www.data.gouv.fr/api/1/datasets/arrets-de-transport-en-france/'
DESTINATION = LAKEHOUSE_FILES / 'bronze' / 'gtfs_national'
DESTINATION.mkdir(parents=True, exist_ok=True)

# Si True, télécharge même si le CSV est déjà stocké et valide.
FORCE_REDOWNLOAD = False

# Tolérance hackathon explicite : si le SHA-1 de la source diffère, autoriser l'usage
# UNIQUEMENT SI la taille correspond EXACTEMENT ET l'en-tête / des lignes CSV sont valides.
# Le résultat sera marqué 'AVERTISSEMENT' dans le manifeste : intégrité non garantie.
# Mettre False pour bloquer systématiquement en cas de SHA-1 différent.
ALLOW_CHECKSUM_MISMATCH_IF_SIZE_AND_CSV_VALID = True

EXPECTED_CSV_COLUMNS = {'stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'dataset_id'}

print('Lakehouse connecté :', LAKEHOUSE_FILES)
print('Dossier national :', DESTINATION)

# %%
# 2 — Métadonnées et sélection fiable de la ressource officielle
response = requests.get(DATASET_API, timeout=40)
response.raise_for_status()
metadata = response.json()

candidates = [
    r for r in metadata.get('resources', [])
    if (r.get('format') or '').lower() == 'csv'
    and 'gtfs_stops_france' in (r.get('title') or '').lower()
    and r.get('url')
]
if not candidates:
    raise RuntimeError('CSV national introuvable dans les métadonnées officielles.')

resource = max(candidates, key=lambda r: r.get('last_modified') or '')
filename = Path(resource['title']).name
if not re.fullmatch(r'[\w.\-]+\.csv', filename, flags=re.IGNORECASE):
    raise ValueError('Nom de ressource inattendu : ' + filename)

# IMPORTANT : privilégier l'URL statique publiée, plutôt que l'API de redirection `latest`.
download_url = resource['url']
file_path = DESTINATION / filename
temp_path = DESTINATION / (filename + '.part')
report_path = DESTINATION / (filename + '.provenance.json')

checksum = resource.get('checksum') or {}
expected_sha1 = (checksum.get('value') or '').lower() if (checksum.get('type') or '').lower() == 'sha1' else None
expected_bytes = resource.get('filesize')
expected_bytes = int(expected_bytes) if expected_bytes is not None else None

print('Fichier :', filename)
print('Source :', download_url)
print('Taille annoncée (octets) :', expected_bytes)
print('SHA-1 publié :', expected_sha1 or 'non disponible')
print('Dernière mise à jour :', resource.get('last_modified'))

# %%
# 3 — Validation du fichier CSV sans le charger dans pandas
def describe_csv(path: Path) -> dict:
    """Vérifie que le fichier ressemble à l'export national de points d'arrêt."""
    with path.open('rb') as raw:
        prefix = raw.read(64_000)
    if not prefix:
        raise ValueError('Fichier vide.')
    if prefix.startswith((b'PK\x03\x04', b'\x1f\x8b')):
        raise ValueError('Un contenu ZIP/GZIP a été reçu au lieu du CSV attendu.')
    if prefix.lstrip().lower().startswith((b'<html', b'<!doctype', b'{"error"', b'{"message"')):
        raise ValueError('La source a retourné une page HTML ou une erreur JSON.')
    try:
        head_text = prefix.decode('utf-8-sig')
    except UnicodeDecodeError as exc:
        raise ValueError('Début du fichier non décodable en UTF-8.') from exc

    header = head_text.splitlines()[0]
    if not header:
        raise ValueError('En-tête CSV vide.')
    try:
        dialect = csv.Sniffer().sniff(head_text[:8192], delimiters=',;\t')
        sep = dialect.delimiter
    except csv.Error:
        sep = max((',', ';', '\t'), key=header.count)

    with path.open('r', encoding='utf-8-sig', newline='') as fh:
        reader = csv.DictReader(fh, delimiter=sep)
        columns = list(reader.fieldnames or [])
        missing = EXPECTED_CSV_COLUMNS - set(columns)
        if missing:
            raise ValueError(f'Colonnes attendues absentes : {sorted(missing)} ; trouvées : {columns}')
        sampled_rows = 0
        rows_with_coordinates = 0
        for row in reader:
            sampled_rows += 1
            try:
                lat = float(row['stop_lat'])
                lon = float(row['stop_lon'])
                if -90 <= lat <= 90 and -180 <= lon <= 180:
                    rows_with_coordinates += 1
            except (ValueError, TypeError):
                pass
            if sampled_rows >= 20:
                break
    if sampled_rows == 0 or rows_with_coordinates == 0:
        raise ValueError('Aucune ligne échantillonnée ne contient des coordonnées valides.')
    return {
        'separator': sep,
        'columns': columns,
        'sample_rows_checked': sampled_rows,
        'sample_rows_with_coordinates': rows_with_coordinates,
    }


def hash_sha1(path: Path) -> str:
    digest = hashlib.sha1()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(4 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

# %%
# 4 — Télécharger en streaming, contrôler, archiver les résultats
http_details = {'url': None, 'content_type': None, 'content_encoding': None}

if file_path.exists() and not FORCE_REDOWNLOAD:
    print('Fichier déjà présent : contrôle du cache plutôt que nouveau téléchargement.')
    candidate_path = file_path
else:
    if temp_path.exists():
        temp_path.unlink()
    print('Téléchargement en cours (environ 437 Mo)...')
    total = 0
    digest = hashlib.sha1()
    # L'en-tête demande des octets non compressés afin de comparer taille et SHA-1.
    with requests.get(
        download_url,
        stream=True,
        timeout=(30, 180),
        headers={'Accept-Encoding': 'identity'},
        allow_redirects=True,
    ) as r:
        r.raise_for_status()
        http_details = {
            'url': r.url,
            'content_type': r.headers.get('Content-Type'),
            'content_encoding': r.headers.get('Content-Encoding'),
        }
        with temp_path.open('wb') as fh:
            for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                if not chunk:
                    continue
                fh.write(chunk)
                digest.update(chunk)
                total += len(chunk)
                if total // (50 * 1024 * 1024) != (total - len(chunk)) // (50 * 1024 * 1024):
                    print(f'  {total / 1048576:.0f} MiB téléchargés...')
    candidate_path = temp_path
    print('Téléchargement terminé :', total, 'octets')

actual_bytes = candidate_path.stat().st_size
actual_sha1 = hash_sha1(candidate_path) if candidate_path == file_path else digest.hexdigest()
size_match = expected_bytes is None or expected_bytes == actual_bytes
sha1_match = expected_sha1 is None or expected_sha1 == actual_sha1

# Vérifier la structure AVANT toute promotion dans le dossier Bronze.
csv_report = describe_csv(candidate_path)

if not size_match:
    raise ValueError(
        f'Taille incorrecte : reçu {actual_bytes} octets ; attendu {expected_bytes}. '
        f'Fichier temporaire conservé pour diagnostic : {candidate_path}'
    )

if sha1_match:
    integrity_status = 'CHECKSUM_OK' if expected_sha1 else 'CHECKSUM_NON_PUBLIE'
else:
    integrity_status = 'AVERTISSEMENT_CHECKSUM_DIFFERENT_TAILLE_ET_STRUCTURE_OK'
    print('\n⚠️ SHA-1 différent des métadonnées officielles.')
    print('Publié :', expected_sha1)
    print('Calculé :', actual_sha1)
    print('La taille et la structure CSV sont vérifiées ; une incohérence reste à investiguer.')
    if not ALLOW_CHECKSUM_MISMATCH_IF_SIZE_AND_CSV_VALID:
        raise ValueError('SHA-1 non conforme : désactiver le mode strict uniquement après vérification.')

if candidate_path == temp_path:
    temp_path.replace(file_path)

report = {
    'dataset': metadata.get('title'),
    'dataset_page': metadata.get('page'),
    'resource_id': resource.get('id'),
    'resource_last_modified': resource.get('last_modified'),
    'license': metadata.get('license'),
    'source_url': download_url,
    'file_path': str(file_path),
    'ingested_at_utc': datetime.now(timezone.utc).isoformat(),
    'expected_bytes': expected_bytes,
    'actual_bytes': actual_bytes,
    'expected_sha1': expected_sha1,
    'actual_sha1': actual_sha1,
    'integrity_status': integrity_status,
    'csv_validation': csv_report,
    'http_details': http_details,
    'limitations': [
        'Export expérimental, pas un registre national exhaustif et actualisé.',
        'Arrêts non dédoublonnés ; stop_id non unique au niveau national.',
        'Couverture des DROM à vérifier dans les données.',
    ],
}
with report_path.open('w', encoding='utf-8') as fh:
    json.dump(report, fh, ensure_ascii=False, indent=2)

print('\n=== IMPORT BRONZE TERMINÉ ===')
print('Fichier :', file_path)
print('Taille (MiB) :', round(actual_bytes / 1048576, 2))
print('Intégrité :', integrity_status)
print('Colonnes :', len(csv_report['columns']))
print('Noms :', csv_report['columns'])
print('Manifeste :', report_path)

# %% [markdown]
# ## Étape suivante (facultative) : prévisualiser avec Spark
# La suite peut être exécutée après confirmation de la réception du fichier.
# Elle conserve les arrêts d'origine, sans dédoublonner artificiellement `stop_id`.

# %%
# 5 — Lecture distribuée avec PySpark (pas de Pandas pour le gros fichier)
from pyspark.sql import functions as F

spark_csv = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'false')
    .option('sep', csv_report['separator'])
    .option('encoding', 'UTF-8')
    .option('mode', 'FAILFAST')
    .csv('Files/bronze/gtfs_national/' + filename)
)

print('Nombre de lignes brutes :', spark_csv.count())
print('Nombre de colonnes :', len(spark_csv.columns))
display(spark_csv.select('dataset_id', 'stop_id', 'stop_name', 'stop_lat', 'stop_lon').limit(10))

# %%
# 6 — Transformation Silver nationale (exécuter si le schéma silver existe)
# Préservation de la provenance pour éviter de confondre les stop_id entre réseaux.
# Aucune déduplication des arrêts physiques à ce stade.
required = ['dataset_id', 'stop_id', 'stop_name', 'stop_lat', 'stop_lon']
missing = set(required) - set(spark_csv.columns)
if missing:
    raise ValueError('Colonnes manquantes : ' + str(sorted(missing)))

clean = (
    spark_csv
    .withColumn('stop_id', F.trim(F.col('stop_id')))
    .withColumn('stop_name', F.trim(F.col('stop_name')))
    .withColumn('stop_lat', F.col('stop_lat').cast('double'))
    .withColumn('stop_lon', F.col('stop_lon').cast('double'))
    .withColumn('source', F.lit('GTFS_NATIONAL_DATA_GOUV'))
    .withColumn('country_code', F.lit('FR'))
    .withColumn('ingested_at_utc', F.current_timestamp())
)

valid = (
    F.col('stop_id').isNotNull() & (F.col('stop_id') != '')
    & F.col('stop_name').isNotNull() & (F.col('stop_name') != '')
    & F.col('stop_lat').between(-90, 90)
    & F.col('stop_lon').between(-180, 180)
)

rejected_count = clean.filter(~F.coalesce(valid, F.lit(False))).count()
clean_valid = clean.filter(F.coalesce(valid, F.lit(False)))

(
    clean_valid.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('silver.france_transport_stops')
)

print('Table créée : silver.france_transport_stops')
print('Lignes retenues :', clean_valid.count())
print('Lignes rejetées (coordonnées/identifiants invalides) :', rejected_count)
print('ATTENTION : les points d\'arrêt peuvent être dupliqués entre plusieurs sources.')
